In [1]:
class GridEnvironment:
    """
    Represents a 2D grid-based warehouse environment.
    
    Attributes:
        grid: 2D array where True = free space, False = obstacle (shelf/wall)
        height: Number of rows
        width: Number of columns
        
    Methods provide obstacle checking, neighbor finding, and visualization.
    """

    def __init__(self, filename):
        """ Load grid from file """
        self.grid = self.load_from_file(filename)
        """ Initialize grid with dimensions """
        self.height = len(self.grid)
        self.width = len(self.grid[0]) if self.height > 0 else 0

    def is_valid_position(self, x, y):
        """ Check if position is within grid bounds """
        return 0 <= x < self.width and 0 <= y < self.height

    def is_walkable(self, x, y):
        """ Check if position is walkable (not obstacle)"""
        return self.is_valid_position(x, y) and self.grid[y][x]

    def get_neighbors(self, x, y):
        """Get adjacent cells (4-directional)"""
        directions = [(0,1), (1,0), (0,-1), (-1,0)]
        neighbors = []

        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if self.is_walkable(nx, ny):
                neighbors.append((nx, ny))

        return neighbors

    def load_from_file(self, filename):
        """ Load grid from file """
        grid = []
        with open(filename, 'r') as f:
            for line in f:
                row = []
                for char in line.strip():
                    if char == '.':
                        row.append(True)
                    elif char == 'T':
                        row.append(False)
                grid.append(row)
        return grid

    def save_to_file(self, filename):
        """ Save grid to file """
        with open(filename, 'w') as f:
            for row in self.grid:
                line = ''.join(['.' if cell else 'T' for cell in row])
                f.write(line + '\n')

    def visualize(self):
        import matplotlib.pyplot as plt
        from matplotlib.colors import ListedColormap
        import numpy as np
        """Visualize the warehouse grid"""
        grid_array = np.array(self.grid, dtype=int)
        
        fig, ax = plt.subplots(figsize=(12, 12))
        
        cmap = ListedColormap(['black', 'white'])
        ax.imshow(grid_array, cmap=cmap, origin='upper', interpolation='nearest')
        
        ax.set_xticks(np.arange(-0.5, self.width, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, self.height, 1), minor=True)
        
        ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.3, alpha=0.5)
        
        ax.set_xticks(np.arange(0, self.width, 20))
        ax.set_yticks(np.arange(0, self.height, 20))
        

        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_title(f"Warehouse Grid ({self.width}x{self.height})")
        
        plt.tight_layout()
        plt.show()

print("Grid class defnined")


Grid class defnined


In [2]:
class Robot :
    """ Represents a single robot """
    def __init__(self, robot_id: int, start_position: tuple, goal_position: tuple, color: str):
        """
        Args:
            grid (GridEnvironment): The grid this robot moves in
            start_pos (tuple): Starting (x, y) position
            goal_pos (tuple): Goal (gx, gy) position
            color (str): Robot color
        """

        self.id = robot_id
        self.start_pos = start_position
        self.goal_pos = goal_position
        self.color = color

    def get_start_position(self) -> tuple:
        """Get start position"""
        return self.start_position

    def get_goal_position(self) -> tuple:
        """Get goal position """
        return self.goal_position

    def get_color(self) -> str:
        """Get color"""
        return self.color
    
print("Robot class defined")

Robot class defined


In [3]:
class Node:
    """
    Represents a node in the search tree.
    
    Attributes:
        state: The current state (configuration of all robots)
        parent: Parent node in search tree
        action: Action that led to this node
        g: Cumulative cost (actual path cost from start)
        h: Heuristic value (estimated cost to goal)
        f: Total evaluation cost (g + h)
        depth: Depth in search tree
    """

    def __init__(self, state, parent=None, action=None, g=0, h=0):
        """
        Initialize a search node.
        
        Args:
            state: State configuration (dict of robot positions at each time)
            parent: Parent node
            action: Action to reach this node
            g: Cost from start to this node
            h: Heuristic estimate to goal
        """
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g  # Actual cost
        self.h = h  # Heuristic estimate
        self.f = g + h  # f(n) = g(n) + h(n)
        self.depth = 0 if parent is None else parent.depth + 1

    def __hash__(self):
        """
        Make node hashable by hashing the state.
        Since state is a dict, convert to a canonical string form.
        """
        return hash(str(sorted(self.state.items()))) # ⚠️⚠️⚠️⚠️⚠️⚠️ We need to consider the list of robots here, lists are imutable

    def __eq__(self, other):
        """
        Two nodes are equal if their states are identical.
        """
        return self.state == other.state    

    def __gt__(self, other):
        """
        Compare this node with another node based on the evaluation function (f).

        Input Parameters:
            - other: Another Node instance.

        Output:
            - True if this node's f is greater than the other's f, else False.
        """
        return isinstance(other, Node) and self.f > other.f
    
    def __repr__(self):
        return f"Node(depth={self.depth}, g={self.g}, h={self.h}, f={self.f})"
    
print("Class Node definied")

Class Node definied


In [4]:
class Candidate:
    """
    Represents a candidate solution in a local search context.

    Attributes:
        state: The specific configuration of the solution (e.g., a tour permutation, queen positions).
        value: The evaluation score of the state (lower is generally better in minimization problems).
    """
    def __init__(self, state, value):
        self.state = state
        self.value = value

    def __repr__(self):
        pass

print("Class Candidate defined")


Class Candidate defined


In [5]:
import copy
class AutomatedWarehouseRobotControllerProblem:
    """
    """
    def __init__(self, initial_state):
        """
        Initialize the MAPF problem.
        
        Args:
            grid: Grid object representing the environment
            robots: List of Robot objects
            max_time: Maximum time steps allowed (prevents infinite loops)
        """
        self.state = initial_state

    def is_goal(self) -> bool:
        pass

    def get_valid_actions(self, state) -> list[dict[int: tuple]]:
        """Returns possible actions for the given state, as th next postions of the robots"""
        pass

    def apply_action(self, state, action):
        """Applies the next moves that are in the action to the given state and returns the new state"""
        pass

    def expand_node(self, node) -> list[Node]:
        """Returns the child nodes of the given node"""
        pass

    def generate_neighbors(self, state):
        """Returns the neighbors of the current state, will be used in local search'Hill climbing optimization'"""
        pass

    def get_makespan(self, state) -> int:
        pass

    def get_flowtime(self, state) -> int:
        pass
    
    # We need functions that detect if the appllying an actions will lead to a deadlock, and to detect the vertex and edge collisions

print("Class AutomatedWarehouseRobotControllerProblem defined")

Class AutomatedWarehouseRobotControllerProblem defined


In [6]:
import heapq
import copy
from collections import defaultdict

# ============================================================
# INDEPENDENT A* ALGORITHM
# ============================================================

class IndependentAStar:
    """
    Independent A* (baseline MAPF approach).
    
    Each robot plans its own shortest path using A* — independently,
    without any awareness of other robots. Paths are then executed
    simultaneously, which may lead to collisions.
    
    This is the simplest MAPF baseline, useful as a benchmark.
    """

    # ----------------------------------------------------------
    # HEURISTIC
    # ----------------------------------------------------------

    def heuristic(self, pos, goal):
        """
        Manhattan distance heuristic: |dx| + |dy|
        Admissible for grid with 4-directional movement (never overestimates).
        """
        return abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])

    # ----------------------------------------------------------
    # A* FOR A SINGLE ROBOT
    # ----------------------------------------------------------

    def a_star_single(self, robot, grid):
        """
        Run A* for one robot, ignoring all other robots.

        Args:
            robot (Robot): The robot to plan for (uses .start_pos and .goal_pos)
            grid (GridEnvironment): The warehouse grid

        Returns:
            list[tuple]: Sequence of (x, y) positions from start to goal (inclusive),
                         or empty list if no path found.
        """
        start = robot.start_pos
        goal  = robot.goal_pos

        # Each entry in the priority queue: (f, g, position)
        # We store g separately so we can reconstruct the cost.
        open_list = []
        heapq.heappush(open_list, (self.heuristic(start, goal), 0, start))

        # came_from maps position → parent position (for path reconstruction)
        came_from = {start: None}

        # g_score[pos] = cheapest known cost to reach pos
        g_score = {start: 0}

        while open_list:
            f, g, current = heapq.heappop(open_list)

            # ── Goal reached ──────────────────────────────────
            if current == goal:
                return self._reconstruct_path(came_from, current)

            # Skip if we already found a better path to current
            if g > g_score.get(current, float('inf')):
                continue

            # ── Expand neighbours (4-directional) ─────────────
            for nx, ny in grid.get_neighbors(*current):
                neighbor = (nx, ny)
                tentative_g = g + 1  # each move costs 1

                if tentative_g < g_score.get(neighbor, float('inf')):
                    g_score[neighbor] = tentative_g
                    came_from[neighbor] = current
                    h = self.heuristic(neighbor, goal)
                    heapq.heappush(open_list, (tentative_g + h, tentative_g, neighbor))

        # No path found
        return []

    def _reconstruct_path(self, came_from, current):
        """Walk backwards through came_from to build the path list."""
        path = []
        while current is not None:
            path.append(current)
            current = came_from[current]
        path.reverse()
        return path

    # ----------------------------------------------------------
    # PLAN PATHS FOR ALL ROBOTS
    # ----------------------------------------------------------

    def plan_paths(self, robots, grid):
        """
        Run A* independently for every robot and store each robot's
        planned path inside the state dictionary.

        Args:
            robots (list[dict]): The 'robots' list from the global state.
                                 Each entry has keys: id, position, goal, path, path_index, at_goal.
            grid (GridEnvironment): The warehouse grid.

        Side-effects:
            - Sets state_robot['path']       = computed list of (x,y) tuples
            - Sets state_robot['path_index'] = 0  (start of path)
            - Sets state_robot['at_goal']    = False (reset)
        """
        for state_robot in robots:
            # Build a lightweight proxy so we can reuse a_star_single
            # without modifying the Robot class.
            class _Proxy:
                pass
            proxy = _Proxy()
            proxy.start_pos = state_robot['position']   # current position is the start
            proxy.goal_pos  = state_robot['goal']

            path = self.a_star_single(proxy, grid)

            if not path:
                print(f"[WARNING] Robot {state_robot['id']}: no path found from "
                      f"{proxy.start_pos} to {proxy.goal_pos}")

            state_robot['path']       = path
            state_robot['path_index'] = 0
            state_robot['at_goal']    = (proxy.start_pos == proxy.goal_pos)

        print("[IndependentAStar] Paths planned for all robots.")

    # ----------------------------------------------------------
    # SIMULATION
    # ----------------------------------------------------------

    def simulate(self, state):
        """
        Execute one full simulation: advance all robots step by step
        along their pre-computed paths until every robot has reached
        its goal (or deadlock is detected).

        Collision detection at each step:
            1. Vertex collision  – two robots occupy the same cell at the same time
            2. Edge collision    – two robots swap positions (pass through each other)

        Deadlock detection:
            If no robot moves for DEADLOCK_PATIENCE consecutive steps, flag deadlock.

        Args:
            state (dict): The global MAPF state (mutated in place).
        """
        DEADLOCK_PATIENCE = 10   # steps with zero movement before we call deadlock

        robots        = state['robots']
        grid          = state['grid']
        no_move_count = 0        # consecutive steps with no movement

        while True:
            # ── Check termination ─────────────────────────────
            if all(r['at_goal'] for r in robots):
                print(f"[Simulation] All robots reached goal at time step {state['time_step']}.")
                break

            if state['deadlock']:
                print(f"[Simulation] Deadlock detected at time step {state['time_step']}.")
                break

            # ── Advance each robot one step ───────────────────
            prev_positions = {r['id']: r['position'] for r in robots}
            next_positions = {}

            for r in robots:
                if r['at_goal']:
                    next_positions[r['id']] = r['position']   # stay put
                    continue

                path = r['path']
                idx  = r['path_index']

                if idx + 1 < len(path):
                    next_pos = path[idx + 1]
                else:
                    # Path exhausted (robot should be at goal)
                    next_pos = r['position']

                next_positions[r['id']] = next_pos

            # ── Vertex collision detection ────────────────────
            # Group robots by their NEXT position
            cell_occupants = defaultdict(list)
            for rid, pos in next_positions.items():
                cell_occupants[pos].append(rid)

            vertex_collisions = {
                cell: rids
                for cell, rids in cell_occupants.items()
                if len(rids) > 1
            }

            # ── Edge collision detection ──────────────────────
            # Robot A moves A→B while Robot B moves B→A simultaneously
            edge_collisions = []
            robot_ids = [r['id'] for r in robots]
            for i in range(len(robot_ids)):
                for j in range(i + 1, len(robot_ids)):
                    id_a, id_b = robot_ids[i], robot_ids[j]
                    if (prev_positions[id_a] == next_positions[id_b] and
                            prev_positions[id_b] == next_positions[id_a]):
                        edge_collisions.append((id_a, id_b))

            # ── Update collision counter ──────────────────────
            collision_count = len(vertex_collisions) + len(edge_collisions)
            state['collisions'] += collision_count

            if collision_count > 0:
                print(f"  [t={state['time_step']+1}] Collisions: "
                      f"{len(vertex_collisions)} vertex, {len(edge_collisions)} edge")

            # ── Apply movement ────────────────────────────────
            any_moved = False
            for r in robots:
                new_pos = next_positions[r['id']]
                if new_pos != r['position']:
                    any_moved = True
                r['position']   = new_pos
                r['path_index'] = min(r['path_index'] + 1, len(r['path']) - 1)
                r['at_goal']    = (new_pos == r['goal'])

            # ── Advance time ──────────────────────────────────
            state['time_step'] += 1

            # ── Deadlock check ────────────────────────────────
            if not any_moved:
                no_move_count += 1
                if no_move_count >= DEADLOCK_PATIENCE:
                    state['deadlock'] = True
            else:
                no_move_count = 0

    # ----------------------------------------------------------
    # METRICS
    # ----------------------------------------------------------

    def get_makespan(self, state):
        """
        Makespan = time when the LAST robot reaches its goal.
        Approximated here as max path length among all robots.
        """
        return max((len(r['path']) - 1) for r in state['robots'] if r['path'])

    def get_flowtime(self, state):
        """
        Flowtime = sum of individual travel times (path lengths) for all robots.
        """
        return sum((len(r['path']) - 1) for r in state['robots'] if r['path'])


# ============================================================
# TEST CLASS
# ============================================================

class IndependentAStarTest:
    """
    Test harness for IndependentAStar.

    Demonstrates:
        1. Normal planning and simulation on a small grid
        2. A collision / near-deadlock scenario
    """

    # ----------------------------------------------------------
    # GRID CREATION
    # ----------------------------------------------------------

    def create_grid_from_string(self, layout):
        """
        Build a minimal GridEnvironment from a list of strings.
        '.' = free, 'T' = obstacle.

        We subclass / monkey-patch GridEnvironment so we don't need a file.
        """
        class InlineGrid:
            def __init__(self, rows):
                self.grid   = [[c == '.' for c in row] for row in rows]
                self.height = len(self.grid)
                self.width  = len(self.grid[0]) if self.height else 0

            def is_valid_position(self, x, y):
                return 0 <= x < self.width and 0 <= y < self.height

            def is_walkable(self, x, y):
                return self.is_valid_position(x, y) and self.grid[y][x]

            def get_neighbors(self, x, y):
                neighbors = []
                for dx, dy in [(0,1),(1,0),(0,-1),(-1,0)]:
                    nx, ny = x+dx, y+dy
                    if self.is_walkable(nx, ny):
                        neighbors.append((nx, ny))
                return neighbors

        return InlineGrid(layout)

    # ----------------------------------------------------------
    # STATE INITIALISATION
    # ----------------------------------------------------------

    def build_initial_state(self, grid, robots_cfg):
        """
        Build the state dict from a grid and a list of
        (id, start, goal, color) tuples.
        """
        robots = []
        for rid, start, goal, color in robots_cfg:
            robots.append({
                'id':         rid,
                'position':   start,
                'goal':       goal,
                'path':       [],
                'path_index': 0,
                'at_goal':    start == goal,
                'color':      color
            })

        return {
            'robots':     robots,
            'time_step':  0,
            'collisions': 0,
            'deadlock':   False,
            'grid':       grid
        }

    # ----------------------------------------------------------
    # PRINTING HELPERS
    # ----------------------------------------------------------

    def print_results(self, state, solver):
        print("\n" + "="*60)
        print("RESULTS")
        print("="*60)

        for r in state['robots']:
            status = "✓ at goal" if r['at_goal'] else "✗ NOT at goal"
            print(f"\n  Robot {r['id']} ({status})")
            print(f"    Path length : {len(r['path'])-1} steps")
            print(f"    Path        : {r['path']}")

        print(f"\n  Total collisions : {state['collisions']}")
        print(f"  Makespan         : {solver.get_makespan(state)} steps")
        print(f"  Flowtime         : {solver.get_flowtime(state)} steps")
        print(f"  Deadlock         : {state['deadlock']}")
        print("="*60)

    # ----------------------------------------------------------
    # TEST 1 – NORMAL SCENARIO
    # ----------------------------------------------------------

    def run_normal_scenario(self):
        """
        10×10 open grid, three robots crossing diagonally.
        Collisions are expected (Independent A* does not avoid them).
        """
        print("\n" + "#"*60)
        print("# TEST 1: Normal crossing scenario (10×10 grid)")
        print("#"*60)

        layout = [
            "..........",
            "..........",
            "..........",
            "..........",
            "..........",
            "..........",
            "..........",
            "..........",
            "..........",
            ".........."
        ]

        grid = self.create_grid_from_string(layout)

        robots_cfg = [
            (1, (1,1), (8,8), 'red'),
            (2, (8,1), (1,8), 'blue'),
            (3, (1,8), (8,1), 'green'),
        ]

        state  = self.build_initial_state(grid, robots_cfg)
        solver = IndependentAStar()

        solver.plan_paths(state['robots'], state['grid'])
        solver.simulate(state)
        self.print_results(state, solver)

    # ----------------------------------------------------------
    # TEST 2 – COLLISION / DEADLOCK SCENARIO
    # ----------------------------------------------------------

    def run_collision_scenario(self):
        """
        Narrow corridor: two robots forced to swap positions.
        This guarantees an edge collision and illustrates why
        Independent A* fails in dense / tight environments.

        Layout (5×3):
            .....
            .....
            .....

        Robot A: (0,1) → (4,1)   [moves right]
        Robot B: (4,1) → (0,1)   [moves left]
        Both will try to pass through the same cells — edge collision guaranteed.
        """
        print("\n" + "#"*60)
        print("# TEST 2: Guaranteed collision in narrow corridor")
        print("#"*60)

        layout = [
            ".....",
            ".....",
            "....."
        ]

        grid = self.create_grid_from_string(layout)

        robots_cfg = [
            (1, (0,1), (4,1), 'red'),
            (2, (4,1), (0,1), 'blue'),
        ]

        state  = self.build_initial_state(grid, robots_cfg)
        solver = IndependentAStar()

        solver.plan_paths(state['robots'], state['grid'])

        print("\n  Planned paths (before simulation):")
        for r in state['robots']:
            print(f"    Robot {r['id']}: {r['path']}")

        solver.simulate(state)
        self.print_results(state, solver)

        print("\n  WHY INDEPENDENT A* FAILS HERE:")
        print("  ─────────────────────────────────────────────────────")
        print("  Each robot plans the shortest path in isolation.")
        print("  Robot 1 plans: (0,1)→(1,1)→(2,1)→(3,1)→(4,1)")
        print("  Robot 2 plans: (4,1)→(3,1)→(2,1)→(1,1)→(0,1)")
        print("  At t=1: Robot 1 moves to (1,1), Robot 2 moves to (3,1)  — ok")
        print("  At t=2: Robot 1 wants (2,1), Robot 2 wants (2,1)        — VERTEX collision")
        print("  At t=2: Robot 1 came from (1,1)↔(3,1) Robot 2           — EDGE collision")
        print("  No re-planning occurs → collisions accumulate.")
        print("  Solution: CBS, ICTS, or prioritised planning.")

    # ----------------------------------------------------------
    # ENTRY POINT
    # ----------------------------------------------------------

    def run_all(self):
        self.run_normal_scenario()
        self.run_collision_scenario()


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    tester = IndependentAStarTest()
    tester.run_all()


############################################################
# TEST 1: Normal crossing scenario (10×10 grid)
############################################################
[IndependentAStar] Paths planned for all robots.
  [t=4] Collisions: 0 vertex, 1 edge
  [t=7] Collisions: 1 vertex, 0 edge
[Simulation] All robots reached goal at time step 14.

RESULTS

  Robot 1 (✓ at goal)
    Path length : 14 steps
    Path        : [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (2, 8), (3, 8), (4, 8), (5, 8), (6, 8), (7, 8), (8, 8)]

  Robot 2 (✓ at goal)
    Path length : 14 steps
    Path        : [(8, 1), (7, 1), (6, 1), (5, 1), (4, 1), (3, 1), (2, 1), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8)]

  Robot 3 (✓ at goal)
    Path length : 14 steps
    Path        : [(1, 8), (1, 7), (1, 6), (1, 5), (1, 4), (1, 3), (1, 2), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1)]

  Total collisions : 2
  Makespan         : 14 steps
  Flowtime         : 4